# CropCop Track B — R07 External Validation

**Claim-producing notebook.** Run only after the four immutable Track-B Kaggle input datasets are attached. Use **GPU T4 x2** and a clean **Save & Run All**. The workflow deliberately uses only `cuda:0`; the second T4 is not a scientific dependency.

Scientific order: authority/replay → source verification → prediction-blind family/overlap audit → candidate seals → prediction firewall → R07 S1/S2/S3 inference → fixed family bootstrap → independent QA → `TRACK_B_CLOSED`.

The consumed V1 test is forbidden. No training, external tuning, candidate shopping, seed replacement, mapped-logit renormalization, or post-result remapping is allowed.

## Kaggle setup

Before saving the claim run, use Kaggle **Dependency Manager** to install the exact versions from `journal_extension/track_b_r07/requirements-trackb.lock.txt`. Internet is not required during the scientific run.

Required input roles are discovered from `TRACKB_INPUT_MANIFEST.json`, so dataset mount slugs may change without editing this notebook.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json, os, shutil, subprocess, sys

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/trackb_r07')
DEVICE = 'cuda:0'

manifests = []
for p in sorted(INPUT_ROOT.glob('**/TRACKB_INPUT_MANIFEST.json')):
    obj = json.loads(p.read_text(encoding='utf-8'))
    manifests.append((obj.get('role'), p, obj))
roles = [r for r, _, _ in manifests]
print('Track-B input roles:', roles)
required = {'core', 'historical_compare', 'gvlid_v5', 'irish_potato'}
if set(roles) != required:
    raise RuntimeError(f'Expected exactly {sorted(required)}, got {sorted(roles)}')
core_path, core = next((p, o) for r, p, o in manifests if r == 'core')
repo_rel = str(core.get('repository_root', '')).strip()
if not repo_rel:
    raise RuntimeError('core TRACKB_INPUT_MANIFEST.json must define repository_root')
REPO_ROOT = (core_path.parent / repo_rel).resolve()
RUNNER = REPO_ROOT / 'journal_extension/scripts/run_trackb_r07.py'
if not RUNNER.is_file():
    raise RuntimeError(f'Track-B runner missing: {RUNNER}')

versions = {
    'torch': metadata.version('torch'),
    'torchvision': metadata.version('torchvision'),
    'numpy': metadata.version('numpy'),
    'Pillow': metadata.version('Pillow'),
    'opencv-python-headless': metadata.version('opencv-python-headless'),
}
expected = {
    'torch': '2.12.1',
    'torchvision': '0.27.1',
    'numpy': '2.5.2',
    'Pillow': '12.3.0',
    'opencv-python-headless': '4.12.0.88',
}
print('Locked package versions:', versions)
if versions != expected:
    raise RuntimeError(f'Dependency lock mismatch. Expected {expected}, got {versions}. Configure Kaggle Dependency Manager before Save & Run All.')

print('Repository root:', REPO_ROOT)
print('Working free GB:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)


## Execute the frozen Track-B controller

This is intentionally one clean process. The runner will refuse to instantiate an R07 classifier on external images until **both** candidate audits are terminal and their seal self-hashes verify.

In [ ]:
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    raise RuntimeError(f'Output directory must be empty for a clean claim run: {OUTPUT_ROOT}')
cmd = [
    sys.executable, str(RUNNER),
    '--input-root', str(INPUT_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--device', DEVICE,
    '--workers', '4',
    '--mode', 'all',
]
print('Launching:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## Terminal evidence

The next cell reads only persisted artifacts written by the controller. A valid run must end with independent QA `PASS` and `TRACK_B_CLOSED`.

In [ ]:
qa = json.loads((OUTPUT_ROOT / 'TRACKB_FINAL_QA.json').read_text(encoding='utf-8'))
closure = json.loads((OUTPUT_ROOT / 'TRACKB_FINAL_CLOSURE.json').read_text(encoding='utf-8'))
packages = json.loads((OUTPUT_ROOT / 'TRACKB_PACKAGE_MANIFEST.json').read_text(encoding='utf-8'))
if qa.get('status') != 'PASS':
    raise RuntimeError('Track-B independent QA did not PASS')
if closure.get('status') != 'TRACK_B_CLOSED':
    raise RuntimeError('Track B did not reach TRACK_B_CLOSED')
print(json.dumps(closure, indent=2, sort_keys=True))
print(json.dumps(packages, indent=2, sort_keys=True))


## Interpretation boundary

`EXT-I` supports only audit-bounded source-independent wording for the candidate's frozen mapped scope. `EXT-S` supports cross-dataset/external-domain stress-test wording only. `EXT-X` produces no claim-making classifier inference. No Track-B outcome establishes 120-class field generalization, agronomic diagnosis/treatment readiness, or device/runtime performance.